In [ ]:
!pip install -q wildlife-datasets timm

In [ ]:
from __future__ import annotations
import math
from pathlib import Path

import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from wildlife_datasets.datasets import AnimalCLEF2026

# ── Config ────────────────────────────────────────────────────────────────────
ROOT   = Path('/kaggle/input/competitions/animal-clef-2026')
OUT    = Path('/kaggle/working/models/finetune')
DEVICE = 'cuda'

TRAIN_CONFIGS = [
    dict(dataset='SeaTurtleID2022',  input_size=224, epochs=10, batch_size=16, lr=1e-5, min_samples=1),
    dict(dataset='LynxID2025',       input_size=224, epochs=15, batch_size=16, lr=1e-5, min_samples=1),
    dict(dataset='SalamanderID2025', input_size=224, epochs=20, batch_size=16, lr=2e-5, min_samples=2),
]

BACKBONE_HUB = 'hf-hub:BVRA/MegaDescriptor-L-384'
EMBED_DIM    = 1536


# ── Dataset ───────────────────────────────────────────────────────────────────
class WildlifeTrainDataset(Dataset):
    def __init__(self, meta: pd.DataFrame, root: Path, transform):
        self.meta = meta.reset_index(drop=True)
        self.root = root
        self.transform = transform
        labels = pd.Categorical(self.meta['identity'])
        self.label_codes = torch.tensor(labels.codes, dtype=torch.long)
        self.num_classes = len(labels.categories)

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        path = self.root / self.meta.loc[idx, 'path']
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.label_codes[idx]


# ── ArcFace loss ──────────────────────────────────────────────────────────────
class ArcFaceLoss(nn.Module):
    def __init__(self, num_classes, embedding_size, margin=0.5, scale=64):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_size))
        nn.init.xavier_uniform_(self.weight)
        self.scale = scale
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th    = math.cos(math.pi - margin)
        self.mm    = math.sin(math.pi - margin) * margin
        self.ce    = nn.CrossEntropyLoss()

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        cosine = F.linear(embeddings, F.normalize(self.weight, dim=1))
        sine   = torch.sqrt(1.0 - torch.clamp(cosine ** 2, max=1.0))
        phi    = cosine * self.cos_m - sine * self.sin_m
        phi    = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine).scatter_(1, labels.view(-1, 1), 1)
        output  = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return self.ce(output * self.scale, labels)


# ── Training loop ─────────────────────────────────────────────────────────────
def build_transforms(size):
    mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    return T.Compose([
        T.Resize((size + 32, size + 32)),
        T.RandomCrop(size),
        T.RandomHorizontalFlip(),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        T.RandomGrayscale(p=0.1),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])


def run(cfg):
    torch.manual_seed(42)
    ds_name = cfg['dataset']
    out_dir = OUT / ds_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # Load metadata
    full = AnimalCLEF2026(str(ROOT), transform=None, load_label=False, check_files=False)
    df   = full.metadata
    mask = (df['dataset'] == ds_name) & (df['split'] == 'train') & df['identity'].notna()
    meta = df[mask].reset_index(drop=True)

    if cfg['min_samples'] > 1:
        counts = meta['identity'].value_counts()
        meta = meta[meta['identity'].isin(counts[counts >= cfg['min_samples']].index)].reset_index(drop=True)

    dataset    = WildlifeTrainDataset(meta, ROOT, build_transforms(cfg['input_size']))
    n_classes  = dataset.num_classes
    print(f'[{ds_name}] images={len(dataset)}  identities={n_classes}')

    loader = DataLoader(dataset, batch_size=cfg['batch_size'], shuffle=True,
                        num_workers=2, pin_memory=True)

    model     = timm.create_model(BACKBONE_HUB, pretrained=True, img_size=cfg['input_size']).to(DEVICE)
    objective = ArcFaceLoss(n_classes, EMBED_DIM).to(DEVICE)

    optimizer = optim.AdamW([
        {'params': model.parameters(),     'lr': cfg['lr']},
        {'params': objective.parameters(), 'lr': cfg['lr'] * 10},
    ], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=cfg['lr'] * 0.01)

    for epoch in range(1, cfg['epochs'] + 1):
        model.train()
        total_loss, n_batches = 0.0, 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = objective(model(imgs), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches  += 1
        scheduler.step()
        print(f'  epoch {epoch}/{cfg["epochs"]}  loss={total_loss/n_batches:.4f}')

        if epoch % 5 == 0:
            torch.save({'model': model.state_dict(), 'embed_dim': EMBED_DIM},
                       out_dir / f'epoch_{epoch}.pth')

    final = out_dir / 'backbone_final.pth'
    torch.save({'model': model.state_dict(), 'embed_dim': EMBED_DIM}, final)
    print(f'Saved -> {final}')


for cfg in TRAIN_CONFIGS:
    run(cfg)